In [2]:
from torchvision import datasets, transforms
import numpy as np
import torch
from torch.utils.data.sampler import SubsetRandomSampler
import torch.nn as nn
import torch.nn.functional as F

Primera muestra: tamaño 64x64, 1k muestras

# 1. Llevar el conjunto de datos al entorno

In [3]:
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

In [4]:
train_data = datasets.ImageFolder('datasets_procesados/size_64x64/sample_1k', transform=transform)

PyTorch entra en ls carpeta, ve las subcarpetas fake y real y las une automáticamente en una sola lista.

In [ ]:
test_size = 0.2
valid_size = 0.2
batch_size = 16 

# For test
num_data = len(train_data)
indices_data = list(range(num_data)) 
np.random.shuffle(indices_data)
split_tt = int(np.floor(test_size * num_data)) 
train_idx, test_idx = indices_data[split_tt:], indices_data[:split_tt]

# AQUI TENEMOS LOS ÍNDICES

# For Valid
num_train = len(train_idx)
np.random.shuffle(train_idx)
split_tv = int(np.floor(valid_size * num_train))
train_final_idx = train_idx[split_tv:]
valid_idx = train_idx[:split_tv]
train_idx = train_final_idx  


# define samplers for obtaining training and validation batches
train_sampler = SubsetRandomSampler(train_idx)
test_sampler = SubsetRandomSampler(test_idx)
valid_sampler = SubsetRandomSampler(valid_idx)


# - Imágenes: bloque numérico de 4 dimensiones que agrupa las fotos del lote: {Forma (Shape):  [batch_size, canales, alto, ancho]
# - Etiquetas:  lista con la clase real que corresponde a cada una de las 16 imágenes del paquete: [0,1,1,0,...]
train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, sampler=train_sampler, num_workers=0)
valid_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, sampler=valid_sampler, num_workers=0)
test_loader = torch.utils.data.DataLoader(train_data, sampler=test_sampler, batch_size=batch_size, num_workers=0)

# variable representing classes of the images
classes = [0, 1]

In [11]:
train_features, train_labels = next(iter(train_loader))
train_features.size()

torch.Size([16, 3, 64, 64])

In [12]:
train_labels.size()

torch.Size([16])

# 2. Inicialización de la CNN

Inicializamos el modelo:

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # convolutional layer
        self.conv1 = nn.Conv2d(3, 16, 5) 
        # max pooling layer
        self.pool = nn.MaxPool2d(2, 2) 
        self.conv2 = nn.Conv2d(16, 32, 5)
        self.dropout = nn.Dropout(0.2) 
        # NOTA
        self.fc1 = nn.Linear(32 * 13 * 13, 256)
        self.fc2 = nn.Linear(256, 84) 
        self.fc3 = nn.Linear(84, 2)
        self.softmax = nn.LogSoftmax(dim=1) 
        
    def forward(self, x): 
       
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.dropout(x) 
        x = x.view(-1, 32 * 13 * 13)
        
        x = F.relu(self.fc1(x))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.softmax(self.fc3(x))
        return x
    
# create a complete CNN
model = Net()


# criterion = torch.nn.CrossEntropyLoss()
criterion = torch.nn.NLLLoss()
# Optimizer: Descenso de Gradiente Estocástico.
optimizer = torch.optim.SGD(model.parameters(), lr = 0.003, momentum= 0.9) # lr = tasa de aprendizaje. momentum: no entiendo bien 

# 3. Entrenar el modelo

Los 6 pasos del entrenamiento por cada lote (Batch):
1. Borrar gradientes (optimizer.zero_grad()):
PyTorch acumula los gradientes por defecto en cada iteración. Si no los borras a cero antes de procesar un nuevo lote, sumará las correcciones del lote actual con las del anterior y el modelo no aprenderá correctamente.

2. Paso hacia adelante / Forward Pass (output = model(data)):
Las imágenes del lote entran a la red y se calcula la predicción actual.

3. Calcular la pérdida / Loss (loss = criterion(output, target)):
Compara la predicción devuelta por la red (output) con la etiqueta real (target) para ver cuánto se ha equivocado.

4. Paso hacia atrás / Backward Pass (loss.backward()):
Calcula las derivadas (gradientes) del error hacia atrás a través de toda la red para saber cuánto ha contribuido cada peso al fallo.

5. Optimización (optimizer.step()):
El optimizador aplica el algoritmo para actualizar y corregir ligeramente todos los pesos de la red.

6. Acumular el error (train_loss += loss.item() * data.size(0)):
Suma el error ponderado del lote para al final del ciclo poder saber la pérdida media de toda la época.

In [14]:
# Definimos la variable manualmente
train_on_gpu = torch.cuda.is_available()

# Si hay GPU, también enviamos el modelo a la tarjeta gráfica
if train_on_gpu:
    print('CUDA disponible. El entrenamiento se ejecutará en la GPU.')
    model.cuda()
else:
    print('CUDA no disponible. El entrenamiento se ejecutará en la CPU.')

CUDA no disponible. El entrenamiento se ejecutará en la CPU.


In [ ]:
# number of epochs to train the model
n_epochs = 5 

valid_loss_min = np.inf # track change in validation loss

for epoch in range(1, n_epochs+1):

    # keep track of training and validation loss
    train_loss = 0.0
    valid_loss = 0.0
    
    # train the model #
    model.train()
    for data, target in train_loader:
        # move tensors to GPU if CUDA is available
        if train_on_gpu:
            data, target = data.cuda(), target.cuda()
        # clear the gradients of all optimized variables
        optimizer.zero_grad()
        # forward pass: compute predicted outputs by passing inputs to the model
        output = model(data)
        # calculate the batch loss
        loss = criterion(output, target)
        # backward pass: compute gradient of the loss with respect to model parameters
        loss.backward()
        # perform a single optimization step (parameter update)
        optimizer.step()
        # update training loss
        train_loss += loss.item()*data.size(0)
        
    # Validating the model
    model.eval()
    for data, target in valid_loader:
        if train_on_gpu:
            data, target = data.cuda(), target.cuda()
        # forward pass: compute predicted outputs by passing inputs to the model
        output = model(data)
        # calculate the batch loss
        loss = criterion(output, target)
        # update average validation loss 
        valid_loss += loss.item()*data.size(0)
    
   
    train_loss = train_loss / len(train_sampler)
    valid_loss = valid_loss / len(valid_sampler)
        
    # print training/validation statistics 
    print('Epoch: {} \tTraining Loss: {:.6f} \tValidation Loss: {:.6f}'.format(
        epoch, train_loss, valid_loss))
    
    # save model if validation loss has decreased
    if valid_loss <= valid_loss_min:
        print('Validation loss decreased ({:.6f} --> {:.6f}).  Saving model ...'.format(
        valid_loss_min,
        valid_loss))
        torch.save(model.state_dict(), 'model_cifar.pt')
        valid_loss_min = valid_loss

Epoch: 1 	Training Loss: 0.693674 	Validation Loss: 0.691004
Validation loss decreased (inf --> 0.691004).  Saving model ...
Epoch: 2 	Training Loss: 0.691154 	Validation Loss: 0.688713
Validation loss decreased (0.691004 --> 0.688713).  Saving model ...
Epoch: 3 	Training Loss: 0.685737 	Validation Loss: 0.684444
Validation loss decreased (0.688713 --> 0.684444).  Saving model ...
Epoch: 4 	Training Loss: 0.677656 	Validation Loss: 0.677111
Validation loss decreased (0.684444 --> 0.677111).  Saving model ...
Epoch: 5 	Training Loss: 0.665830 	Validation Loss: 0.679557


# 4. Evaluación del modelo

In [16]:
# 1. Variables de control
test_loss = 0.0
class_correct = [0.0, 0.0]
class_total = [0.0, 0.0]
classes = ['fake', 'real']
batch_size = 16  # Cambiar este número si el batch_size es diferente

model.eval() # Modo evaluación

# Desactivamos el cálculo de gradientes para ahorrar memoria y acelerar la evaluación
with torch.no_grad():
    for data, target in test_loader:
        if train_on_gpu:
            data, target = data.cuda(), target.cuda()
            
        output = model(data)
        loss = criterion(output, target)
        test_loss += loss.item() * data.size(0)
        
        # Obtener la clase predicha
        _, pred = torch.max(output, 1)
        correct_tensor = pred.eq(target.data.view_as(pred))
        
        if train_on_gpu:
            correct = np.squeeze(correct_tensor.cpu().numpy())
        else:
            correct = np.squeeze(correct_tensor.numpy())
            
        # Acumular aciertos por clase
        for i in range(len(target)):
            label = target.data[i]
            class_correct[label] += correct[i].item() if correct.ndim > 0 else correct.item()
            class_total[label] += 1

# 2. Calcular pérdidas y precisión globales
# test_loss = test_loss / len(test_loader.dataset)
test_loss = test_loss / len(test_sampler)
print(f'Test Loss: {test_loss:.6f}\n')

# Precisiones por cada clase individual
for i in range(2):
    if class_total[i] > 0:
        acc = 100 * class_correct[i] / class_total[i]
        print(f'Test Accuracy of {classes[i]:>5s}: {acc:2.0f}% ({int(class_correct[i])}/{int(class_total[i])})')
    else:
        print(f'Test Accuracy of {classes[i]:>5s}: N/A (sin ejemplos)')

# Precisión global total
total_correct = np.sum(class_correct)
total_samples = np.sum(class_total)
overall_acc = 100. * total_correct / total_samples
print(f'\nTest Accuracy (Overall): {overall_acc:2.0f}% ({int(total_correct)}/{int(total_samples)})')

Test Loss: 0.663422

Test Accuracy of  fake: 31% (58/190)
Test Accuracy of  real: 85% (178/210)

Test Accuracy (Overall): 59% (236/400)
